# Brick 8b - XGBoost vs LightGBM vs RandomForest vs ExtraTrees vs MLP

All five classifiers live in `src/brick8_models.py` (`make_estimator`), are tuned on the validation partition in Brick 8, and are saved/compared by `src/brick8b_compare.py`.

This notebook shows:

1. the model definitions and their tuned hyperparameters,
2. how to reproduce the held-out comparison,
3. the final metrics table on the **test partition** of Split G (unseen drugs) and Split R (random split), for both label modes.

Protocol: every model is refit on the split's full training partition, then scored once on that split's test partition. Non-MLP models use balanced class weights and weak-pair weight 0.5; MLP uses early stopping. Nothing is selected on the test data.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
FIGURES = ROOT / "figures"
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


## 1. Model definitions

The exact constructors used in every run. Tree ensembles get balanced class weights; MLP is a scaled two-layer perceptron.

In [ ]:
from brick8_models import make_estimator

for name in ("XGBoost", "LightGBM", "RandomForest", "ExtraTrees", "MLP"):
    est = make_estimator(name, balanced=(name != "MLP"))
    print(f"{name}: {est}")


## 2. Tuned hyperparameters

Produced by the Brick 8 RandomizedSearchCV on the validation partition (PredefinedSplit). The comparison script refits exactly these settings.

In [ ]:
for mode in ("S", "P"):
    info = json.loads((REPORTS / f"brick8_models_{mode}.json").read_text(encoding="utf-8"))
    print(f"=== Mode {mode} === best on validation: {info['best_model']}")
    for name, entry in info["tuning"].items():
        print(f"  {name:12s} val_macroF1={entry['val']['macro_f1']:.4f}  {entry['best_params']}")


## 3. Reproduce the held-out comparison

Runs `src/brick8b_compare.py`, which trains all five models on both splits and both modes. Mode P has ~204k training rows, so it takes a few minutes; uncomment it if you want to rerun both.

In [ ]:
env = {**os.environ, "PYTHONPATH": str(ROOT / "src"), "PYTHONIOENCODING": "utf-8"}

for mode in ("S",):  # add "P" to rerun the paper-like mode as well
    print(f"--- Mode {mode} ---")
    subprocess.run(
        [sys.executable, str(ROOT / "src" / "brick8b_compare.py"), "--mode", mode],
        cwd=ROOT, env=env, check=True,
    )


## 4. Final metrics on held-out test data

**Split G** = drugs unseen during training (the honest headline). **Split R** = random split (paper-like reference). `neg_recall` is recall for class 0 (avoid together); `selective_accuracy` only counts predictions whose top probability clears 0.60.

In [ ]:
def show_table(mode, split):
    path = REPORTS / f"model_comparison_{mode}.csv"
    df = pd.read_csv(path)
    sub = df[df["split"] == split].copy()
    cols = ["model", "accuracy", "balanced_accuracy", "macro_f1",
            "neg_precision", "neg_recall", "nonsig_recall",
            "coverage_at_threshold", "selective_accuracy", "seconds"]
    sub = sub[cols].set_index("model").loc[["XGBoost", "LightGBM", "RandomForest", "ExtraTrees", "MLP"]]
    display(Markdown(f"**Mode {mode} - {split}**"))
    display(sub.style.format({
        c: "{:.4f}" for c in cols[1:-1]
    }).background_gradient(subset=["macro_f1"], cmap="Greens"))

for mode in ("S", "P"):
    if (REPORTS / f"model_comparison_{mode}.csv").exists():
        show_table(mode, "Split G")
        show_table(mode, "Split R")


## 5. Class-level detail and confusion matrices

Class 1 has very few real pairs (24 in Mode S), so its F1 is noisy; class 0 (avoid) and class 2 (non-significant) carry the macro average.

In [ ]:
for mode in ("S", "P"):
    path = REPORTS / f"model_comparison_{mode}.json"
    if not path.exists():
        continue
    details = json.loads(path.read_text(encoding="utf-8"))
    for split, payload in details["splits"].items():
        print(f"### Mode {mode} - {split} (test rows {payload['test_rows']:,}, drugs {payload['test_drugs']})")
        for name, m in payload["models"].items():
            cm = pd.DataFrame(m["confusion_matrix"], index=["true 0", "true 1", "true 2"],
                              columns=["pred 0", "pred 1", "pred 2"])
            print(f"{name}: macroF1={m['macro_f1']:.4f} neg_recall={m['per_class'][0]['recall']:.4f}")
            print(cm.to_string())
            print()


## 6. Figures

In [ ]:
for mode in ("S", "P"):
    fig = FIGURES / f"model_comparison_{mode}.png"
    if fig.exists():
        display(Markdown(f"**Mode {mode}**"))
        display(Image(filename=str(fig)))


## 7. Run from a terminal

```bash
cd ..
PYTHONPATH=src python src/brick8b_compare.py --mode S
PYTHONPATH=src python src/brick8b_compare.py --mode P
```

Artifacts: `reports/model_comparison.csv`, `reports/model_comparison_{S,P}.csv|json|md`, `figures/model_comparison_{S,P}.png`. The script only writes new files, so existing models, reports, and the web app are unaffected.